# vitalPeriodic

"Periodic" data refers to data which is consistently interfaced from bedside vital signs monitors into eCareManager. Data are generally interfaced as 1 minute averages, and archived into the vitalPeriodic table as 5 minute median values.

The vital signs are **not** validated by care staff, so data quality can vary, but the use of 1 minute averages followed by 5 minute medians before archival removes many spurious readings.

In [ ]:
import sys


In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa+
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from loguru import logger

from notebooks_dani.eicu_db_connection import DatabaseConnection


hv.extension("bokeh")

%matplotlib inline

In [ ]:
entire_dataset = True # plot save flag and csv file decision

In [ ]:
notebook_name = "vitalperiodic_exploration"

plot_outputs_dir_small = "subset"
plot_outputs_dir_all = "all"

#INTER_DATA_PATH = "../../intermediary_data/"
INTER_DATA_PATH = Path("..") / Path("..") / Path("intermediary_data")

print(f"Loading and saving data from/to {Path(INTER_DATA_PATH).resolve()}")

In [ ]:
connection = DatabaseConnection()
engine = connection.engine
query_schema = connection.query_schema

In [ ]:
conn = engine.connect().execution_options(
        stream_results=True)

In [ ]:
df_apache_valid = pd.read_csv(filepath_or_buffer="../intermediary_data/apache_aps_var_results_valid_mortality.csv")
df_apache_valid = df_apache_valid.drop(columns=["Unnamed: 0", "Unnamed: 0_x", "Unnamed: 0_y"])
df_apache_valid.head(n=3)

# Getting the vitalperiodic data for patients with valid apache mortality prediction

In [ ]:
patient_stays_valid: list = df_apache_valid["patientunitstayid"].tolist()

Those are the patientunitstayids with a valid apache score (predictedmortality != -1)

query_valid_apache = query_schema + f"""
SELECT * 
FROM vitalperiodic
WHERE patientunitstayid in ({", ".join(map(str, patient_stays_valid))}) 
ORDER BY patientunitstayid, observationoffset
LIMIT 50000
"""

In [ ]:
#df_vp_apache_valid = pd.read_sql_query(query_valid_apache, engine)

In [ ]:
#vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_50000.csv")
#vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_all.csv")

#if vp_apache_path.is_file():
#    df_vp_apache_valid = pd.read_csv(vp_apache_path, engine="pyarrow")
#    logger.info(f"Loading data from local csv.")
#else:
#    df_vp_apache_valid = pd.read_sql_query(query_valid_apache, engine)
#    df_vp_apache_valid.to_csv(path_or_buf=vp_apache_path)
#    logger.info(f"Loading data from database.")

In [ ]:
#vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_50000.csv")
vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_all.parquet")

if vp_apache_path.is_file():
    df_vp_apache_valid = pd.read_parquet(vp_apache_path, engine="pyarrow")
    #df_vp_apache_valid = pd.read_csv(vp_apache_path, engine="pyarrow")
    logger.info(f"Loading data from local file")
else:
    logger.error(f"File not found")
    df_vp_apache_valid = None

In [ ]:
df_vp_apache_valid_grouped = df_vp_apache_valid.groupby("patientunitstayid")
df_vp_apache_valid_grouped.head()

In [ ]:
unique_stays = df_vp_apache_valid["patientunitstayid"].unique()
print(f"We got {len(unique_stays)} unique patientunitids with an Apache score and the vitalperiodic table in the dataset")

This number is different from the number of valid Apache Scores:

In [ ]:
print(f"Number of patientunitstayids with valid apache predictions {len(patient_stays_valid)}")
print(f"{len(patient_stays_valid) - len(unique_stays)} patientunitstayids are missing in the vitalperiodic table" )

# Determining the coverage of the variables with regard to the patientunitstayid measurements.
A measurement shall be a row of a unique patientunitstayid.

 für jede patientunistayid
    # Anzahl der rows von der gruppe
    # für jedes features
        # Statistische Beschreibung für jedes Feature
        # zähle die Anzahl an fehlenden Samples oder ungültigen
        # berechnen #gültige werte/#messungen



In [ ]:
vp_parameters = list()
for column in df_vp_apache_valid.columns:
    if column == "vitalperiodicid" or column == "patientunitstayid" or column == "observationoffset" or column == "":
        pass
    else:
        vp_parameters.append(column)
        

parameter_sample_count_df = pd.DataFrame(data=parameter_sample_count_list_all)
parameter_coverage_ratio_df = pd.DataFrame(data=parameter_coverage_ratio_list_all)

#sns.heatmap(parameter_sample_count_df.drop(["patientunitstayid", "measurements", "observationoffset"], axis=1), annot=False)

In [ ]:
def generate_parameter_stats(df_vp_apache_valid_grouped, vp_parameters):
    measurement_id_count = dict()

    for group_id, group_df in df_vp_apache_valid_grouped:
        stay_id = group_id
        number_measurements = len(group_df)
        measurement_id_count.update({stay_id: number_measurements})

        parameter_sample_count = {"patientunitstayid": stay_id}
        parameter_coverage_ratio = {"patientunitstayid": stay_id}
        parameter_missing_count = {"patientunitstayid": stay_id}

        for param in vp_parameters:
            count = group_df[param].count()
            coverage_ratio = count / number_measurements
            
            count_missing = number_measurements - count
            
            parameter_sample_count[param] = count
            parameter_coverage_ratio[param] = coverage_ratio
            parameter_missing_count[param] = count_missing

        yield parameter_sample_count, parameter_coverage_ratio, parameter_missing_count


In [ ]:
parameter_sample_count_list_all = []
parameter_missing_count_list_all = []
parameter_coverage_ratio_list_all = []


for sample_count, coverage_ratio, missing_count in generate_parameter_stats(df_vp_apache_valid_grouped, vp_parameters):
    parameter_sample_count_list_all.append(sample_count)
    parameter_missing_count_list_all.append(missing_count)
    parameter_coverage_ratio_list_all.append(coverage_ratio)

parameter_sample_count_df = pd.DataFrame(data=parameter_sample_count_list_all)
parameter_missing_count_df = pd.DataFrame(data=parameter_missing_count_list_all)
parameter_coverage_ratio_df = pd.DataFrame(data=parameter_coverage_ratio_list_all)


In [ ]:

parameter_sample_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_sample_count.csv")
parameter_missing_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_missing_count.csv")
parameter_coverage_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_coverage_count.csv")

if entire_dataset:
    parameter_sample_count_df.to_csv(path_or_buf=parameter_sample_count_path)
    parameter_missing_count_df.to_csv(path_or_buf=parameter_missing_count_path)
    parameter_coverage_ratio_df.to_csv(path_or_buf=parameter_coverage_count_path)


In [ ]:
number_stays = parameter_coverage_ratio_df.shape[0]
number_stays

In [ ]:
fig_sample, ax_sample = plt.subplots(1,1, figsize=(15,15))


sns.heatmap(parameter_sample_count_df.drop(["patientunitstayid"], axis=1), ax=ax_sample, annot=False)

ax_sample.collections[0].colorbar.set_label("Number of Samples")
plt.title(f"{number_stays} ICU stays; number of samples per variable")

filename_corr = "number_of_samples_per_variable_heatmap"
notebook_name = "vitalperiodic_exploration"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
fig_cov_rat, ax_cov_rat = plt.subplots(1,1, figsize=(15,15))

sns.heatmap(parameter_coverage_ratio_df.drop(["patientunitstayid"], axis=1), ax=ax_cov_rat, annot=False)

ax_cov_rat.collections[0].colorbar.set_label("Ratio")
plt.title(f"{number_stays} ICU stays; #samples/#measurements")


filename_corr = "ratio_samples_measurement_per_variable_heatmap"
notebook_name = "vitalperiodic_exploration"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
fig_missing, ax_missing = plt.subplots(1,1, figsize=(15,15))

sns.heatmap(parameter_missing_count_df.drop(["patientunitstayid"], axis=1), ax=ax_missing, annot=False)

ax_missing.collections[0].colorbar.set_label("Number")
plt.title(f"{number_stays} ICU stays; #missing values")


filename_corr = "number_missing_values_per_variable_heatmap"
notebook_name = "vitalperiodic_exploration"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

# Descriptive Statistics of Vitalperiodic with valid Apache predictions

In [ ]:
vp_apache_stats = df_vp_apache_valid[vp_parameters].describe()

In [ ]:
print(vp_apache_stats)


In [ ]:
vp_apache_stats_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_stats.csv")

if entire_dataset:
    vp_apache_stats.to_csv(path_or_buf=vp_apache_stats_path)
    

# Histograms and Boxplots


## Histograms with KDE

In [ ]:
fig_hists, axs_hists = plt.subplots(ncols=3, nrows=len(vp_parameters) // 3 + 1, figsize=(10,10))

for idx, param in enumerate(vp_parameters):
    sns.histplot(df_vp_apache_valid[param], ax=axs_hists.flat[idx], kde=True)
    plt.suptitle(f"Histogram for {param}")

fig_hists.suptitle("Histograms Vitalperiodic")
fig_hists.tight_layout()


filename_corr = "histograms vitalperiodic"

if entire_dataset:
    plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(plot_outputs_dir_all) / Path(filename_corr)
else:
    plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(plot_outputs_dir_small) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

## Boxplots for multiple quantiles

In [ ]:
fig_hists, axs_hists = plt.subplots(ncols=3, nrows=len(vp_parameters) // 3 + 1, figsize=(10,10))



for idx, param in enumerate(vp_parameters):
    sns.boxenplot(df_vp_apache_valid[param], ax=axs_hists.flat[idx], fill=False)
    #plt.suptitle(f"Boxplot for {param}")
    

fig_hists.suptitle("Boxplots Vitalperiodic")
fig_hists.tight_layout()

filename_corr = "boxplots vitalperiodic"

if entire_dataset:
    plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(plot_outputs_dir_all) / Path(filename_corr)
else:
    plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(plot_outputs_dir_small) / Path(filename_corr)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

## Testing for normality and skewness


### Kolmogorov-Smirnov Normality Test

In [ ]:
from scipy import stats
import numpy as np

kstest_results = list()

for param in vp_parameters:
    score, p_val = stats.kstest(df_vp_apache_valid[param].dropna(), cdf=stats.norm.cdf)
    kstest_results.append({ "score": score, "p_value": p_val})
    
    
df_kstest_results = pd.DataFrame(data=kstest_results, index=vp_parameters)

vp_apache_kstest_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_kstest.csv")

if entire_dataset:
    df_kstest_results.to_csv(path_or_buf=vp_apache_kstest_path)


In [ ]:
print(df_kstest_results)

**KS Test: Reject the H_0 (normal distribution of the data) if  the p-value is smaller that 0.05**

### Skewness

In [ ]:
skewness_results = list()

for param in vp_parameters:
    skewness = stats.skew(df_vp_apache_valid[param], axis=0, nan_policy="omit")
    skewness_results.append({"skewness": skewness})


df_skewness_results = pd.DataFrame(data=skewness_results, index=vp_parameters)

vp_apache_skewness_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_skewness.csv")

if entire_dataset:
    df_skewness_results.to_csv(path_or_buf=vp_apache_skewness_path)

print(df_skewness_results)

**A skewness close to 0 indicates normality**